# Excel Ingestion Demo

## Flow

Excel file → Load workbook → Detect sheet names → Read selected sheet → Clean column names → Detect missing values → Remove duplicates → Save staging output → Save clean output → Generate ingestion log

## 1. Import Required Libraries

In [1]:
import pandas as pd
import json
import uuid
import re
import shutil
from pathlib import Path
from datetime import datetime, timezone

## 2. Define Project Paths

In [2]:
def find_project_root(current_path: Path) -> Path:
    """
    Find project root by walking upward until the data/ folder is found.
    This makes the notebook work even if it is executed from notebooks/data_team/.
    """
    current_path = current_path.resolve()

    for path in [current_path] + list(current_path.parents):
        if (path / "data").exists():
            return path

    raise FileNotFoundError(
        "Could not find project root. Please make sure a 'data/' folder exists in the project."
    )


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

input_path = PROJECT_ROOT / "data" / "sample_inputs" / "inventory.xlsx"

raw_dir = PROJECT_ROOT / "data" / "raw" / "excel"
staging_dir = PROJECT_ROOT / "data" / "staging" / "excel"
clean_dir = PROJECT_ROOT / "data" / "clean" / "excel"
log_dir = PROJECT_ROOT / "logs"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = raw_dir / "inventory_raw.xlsx"
staging_output_path = staging_dir / "sample_excel_staging.csv"
clean_output_path = clean_dir / "sample_excel_clean.csv"
log_output_path = log_dir / "excel_ingestion_log.json"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Input path:", input_path)
print("Input exists:", input_path.exists())
print("Raw output:", raw_output_path)
print("Staging output:", staging_output_path)
print("Clean output:", clean_output_path)
print("Log output:", log_output_path)

Current dir: f:\data\new\quanskill\DataVision_Duy\week2\notebooks\data_team
Project root: F:\data\new\quanskill\DataVision_Duy\week2
Input path: F:\data\new\quanskill\DataVision_Duy\week2\data\sample_inputs\inventory.xlsx
Input exists: True
Raw output: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\excel\inventory_raw.xlsx
Staging output: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\excel\sample_excel_staging.csv
Clean output: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\excel\sample_excel_clean.csv
Log output: F:\data\new\quanskill\DataVision_Duy\week2\logs\excel_ingestion_log.json


## 3. Validate Input File

In [3]:
if not input_path.exists():
    raise FileNotFoundError(f"Input Excel file not found: {input_path}")

if input_path.stat().st_size == 0:
    raise ValueError(f"Input Excel file is empty: {input_path}")

print("Input file validation passed.")

Input file validation passed.


## 4. Helper Functions

In [4]:
def clean_column_name(column_name: str) -> str:
    """
    Convert column names to clean snake_case format.

    Examples:
    - Product ID -> product_id
    - Purchase/Stock in -> purchase_stock_in
    - Cost Price Per Unit (USD) -> cost_price_per_unit_usd
    """
    column_name = str(column_name).strip().lower()
    column_name = column_name.replace("\n", " ")
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    column_name = column_name.strip("_")
    return column_name


def find_header_row(
    excel_path: Path,
    sheet_name: str,
    keywords=None,
    max_rows: int = 30
) -> int:
    """
    Detect the header row index by scanning the first rows for known keywords.
    Returns zero-based row index for pandas header usage.
    """
    if keywords is None:
        keywords = ["Product ID", "Inventory ID"]

    preview = pd.read_excel(
        excel_path,
        sheet_name=sheet_name,
        header=None,
        nrows=max_rows
    )

    for row_idx in range(len(preview)):
        row_values = preview.iloc[row_idx].astype(str).str.strip().tolist()

        for keyword in keywords:
            if keyword in row_values:
                return row_idx

    raise ValueError(f"Could not detect header row using keywords: {keywords}")


def select_best_sheet(sheet_names):
    """
    Select expected inventory sheet if available; otherwise use the first sheet.
    """
    preferred_sheets = [
        "Inventory Records Data",
        "Inventory list",
        "inventory",
        "Sheet1"
    ]

    for sheet in preferred_sheets:
        if sheet in sheet_names:
            return sheet

    return sheet_names[0]

## 5. Start Ingestion Run

In [5]:
run_id = str(uuid.uuid4())
source_name = "inventory_excel"
source_type = "excel"
owner = "Nguyen Minh Duy"

start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Start time:", start_time)

Run ID: 3b802d1b-19b6-4899-85c9-43196e2391a6
Start time: 2026-06-01T04:33:53.123498+00:00


## 6. Load Workbook and Detect Sheet Names

In [6]:
try:
    excel_file = pd.ExcelFile(input_path)
    sheet_names = excel_file.sheet_names

    print("Excel workbook loaded successfully.")
    print("Detected sheets:", sheet_names)

    status = "success"
    error_message = None

except Exception as error:
    status = "failed"
    error_message = str(error)
    raise

Excel workbook loaded successfully.
Detected sheets: ['Inventory Records Data']


## 7. Select Sheet

In [7]:
selected_sheet = select_best_sheet(sheet_names)

print("Selected sheet:", selected_sheet)

Selected sheet: Inventory Records Data


## 8. Detect Header Row

In [8]:
header_row_index = find_header_row(
    excel_path=input_path,
    sheet_name=selected_sheet,
    keywords=["Product ID", "Inventory ID"],
    max_rows=30
)

print("Detected header row index:", header_row_index)
print("Excel row number:", header_row_index + 1)

Detected header row index: 5
Excel row number: 6


## 9. Read Selected Sheet

In [9]:
df_raw = pd.read_excel(
    input_path,
    sheet_name=selected_sheet,
    header=header_row_index
)

# Drop fully empty rows and columns.
df_raw = df_raw.dropna(axis=0, how="all")
df_raw = df_raw.dropna(axis=1, how="all")

# Remove possible repeated header rows inside the table.
if "Product ID" in df_raw.columns:
    df_raw = df_raw[df_raw["Product ID"].astype(str).str.strip() != "Product ID"]

records_read = len(df_raw)

print("Sheet loaded successfully.")
print("Records read:", records_read)
print("Columns:", df_raw.columns.tolist())

df_raw.head(5)

Sheet loaded successfully.
Records read: 46
Columns: ['Product ID', 'Product Name', 'Opening \nStock', 'Purchase/\nStock in', 'Number of \nUnits Sold', 'Hand-In-\nStock', 'Cost Price \nPer Unit (USD)', 'Cost Price\nTotal (USD)']


,Product ID,Product Name,Opening \nStock,Purchase/\nStock in,Number of \nUnits Sold,Hand-In-\nStock,Cost Price \nPer Unit (USD),Cost Price\nTotal (USD)
0,P101,Laptop,50,20,10,60,1200,72000
1,P102,Monitor,40,15,5,50,500,25000
2,P103,Keyboard,60,25,15,70,50,3500
3,P104,Headphones,30,10,3,37,100,3700
4,P105,Smartphone,70,30,20,80,900,72000


## 10. Save Raw Excel Copy

In [10]:
# Raw layer should preserve the original Excel file.
shutil.copy2(input_path, raw_output_path)

print("Raw Excel file copied to:", raw_output_path)

Raw Excel file copied to: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\excel\inventory_raw.xlsx


## 11. Clean Column Names

In [11]:
df_staging = df_raw.copy()

original_columns = df_staging.columns.tolist()
cleaned_columns = [clean_column_name(col) for col in original_columns]

df_staging.columns = cleaned_columns

print("Original columns:")
print(original_columns)

print("\nCleaned columns:")
print(cleaned_columns)

df_staging.head()

Original columns:
['Product ID', 'Product Name', 'Opening \nStock', 'Purchase/\nStock in', 'Number of \nUnits Sold', 'Hand-In-\nStock', 'Cost Price \nPer Unit (USD)', 'Cost Price\nTotal (USD)']

Cleaned columns:
['product_id', 'product_name', 'opening_stock', 'purchase_stock_in', 'number_of_units_sold', 'hand_in_stock', 'cost_price_per_unit_usd', 'cost_price_total_usd']


,product_id,product_name,opening_stock,purchase_stock_in,number_of_units_sold,hand_in_stock,cost_price_per_unit_usd,cost_price_total_usd
0,P101,Laptop,50,20,10,60,1200,72000
1,P102,Monitor,40,15,5,50,500,25000
2,P103,Keyboard,60,25,15,70,50,3500
3,P104,Headphones,30,10,3,37,100,3700
4,P105,Smartphone,70,30,20,80,900,72000


## 12. Save Parsed Output to Staging

In [12]:
df_staging.to_csv(staging_output_path, index=False, encoding="utf-8")

print("Staging output saved to:", staging_output_path)

Staging output saved to: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\excel\sample_excel_staging.csv


## 13. Check Duplicate Rows

In [13]:
duplicate_count = int(df_staging.duplicated().sum())

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


## 14. Check Missing Values

In [14]:
missing_values = df_staging.isna().sum()
total_missing_values = int(missing_values.sum())

print("Missing values by column:")
print(missing_values)

print("\nTotal missing values:", total_missing_values)

Missing values by column:
product_id                 0
product_name               0
opening_stock              0
purchase_stock_in          0
number_of_units_sold       0
hand_in_stock              0
cost_price_per_unit_usd    0
cost_price_total_usd       0
dtype: int64

Total missing values: 0


## 15. Create Clean Output

In [15]:
df_clean = df_staging.drop_duplicates().copy()

# Drop rows where all fields are empty after parsing.
df_clean = df_clean.dropna(axis=0, how="all")

records_valid = len(df_clean)
records_invalid = records_read - records_valid

print("Records read:", records_read)
print("Records valid after cleaning:", records_valid)
print("Records invalid / removed:", records_invalid)

df_clean.head()

Records read: 46
Records valid after cleaning: 46
Records invalid / removed: 0


,product_id,product_name,opening_stock,purchase_stock_in,number_of_units_sold,hand_in_stock,cost_price_per_unit_usd,cost_price_total_usd
0,P101,Laptop,50,20,10,60,1200,72000
1,P102,Monitor,40,15,5,50,500,25000
2,P103,Keyboard,60,25,15,70,50,3500
3,P104,Headphones,30,10,3,37,100,3700
4,P105,Smartphone,70,30,20,80,900,72000


## 16. Save Cleaned Output

In [16]:
df_clean.to_csv(clean_output_path, index=False, encoding="utf-8")

print("Clean output saved to:", clean_output_path)

Clean output saved to: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\excel\sample_excel_clean.csv


## 17. Generate Ingestion Log

In [17]:
end_time = datetime.now(timezone.utc).isoformat()

ingestion_log = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "input_path_or_url": str(input_path),
    "start_time": start_time,
    "end_time": end_time,
    "status": status,
    "records_read": int(records_read),
    "records_valid": int(records_valid),
    "records_invalid": int(records_invalid),
    "duplicate_rows_removed": int(duplicate_count),
    "missing_values": missing_values.astype(int).to_dict(),
    "total_missing_values": total_missing_values,
    "sheet_names": sheet_names,
    "selected_sheet": selected_sheet,
    "detected_header_row_index": int(header_row_index),
    "detected_excel_header_row_number": int(header_row_index + 1),
    "error_message": error_message,
    "raw_output_path": str(raw_output_path),
    "staging_output_path": str(staging_output_path),
    "clean_output_path": str(clean_output_path),
    "owner": owner
}

with open(log_output_path, "w", encoding="utf-8") as file:
    json.dump(ingestion_log, file, indent=4, ensure_ascii=False)

print("Ingestion log saved to:", log_output_path)
ingestion_log

Ingestion log saved to: F:\data\new\quanskill\DataVision_Duy\week2\logs\excel_ingestion_log.json


{'run_id': '3b802d1b-19b6-4899-85c9-43196e2391a6',
 'source_name': 'inventory_excel',
 'source_type': 'excel',
 'input_path_or_url': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\sample_inputs\\inventory.xlsx',
 'start_time': '2026-06-01T04:33:53.123498+00:00',
 'end_time': '2026-06-01T04:34:27.445727+00:00',
 'status': 'success',
 'records_read': 46,
 'records_valid': 46,
 'records_invalid': 0,
 'duplicate_rows_removed': 0,
 'missing_values': {'product_id': 0,
  'product_name': 0,
  'opening_stock': 0,
  'purchase_stock_in': 0,
  'number_of_units_sold': 0,
  'hand_in_stock': 0,
  'cost_price_per_unit_usd': 0,
  'cost_price_total_usd': 0},
 'total_missing_values': 0,
 'sheet_names': ['Inventory Records Data'],
 'selected_sheet': 'Inventory Records Data',
 'detected_header_row_index': 5,
 'detected_excel_header_row_number': 6,
 'error_message': None,
 'raw_output_path': 'F:\\data\\new\\quanskill\\DataVision_Duy\\week2\\data\\raw\\excel\\inventory_raw.xlsx',
 'staging_output_pa

## 18. Final Output Check

In [18]:
print("Expected outputs:")

print("Raw output exists:", raw_output_path.exists())
print("Staging output exists:", staging_output_path.exists())
print("Clean output exists:", clean_output_path.exists())
print("Log output exists:", log_output_path.exists())

print("\nOutput paths:")
print("Raw:", raw_output_path)
print("Staging:", staging_output_path)
print("Clean:", clean_output_path)
print("Log:", log_output_path)

Expected outputs:
Raw output exists: True
Staging output exists: True
Clean output exists: True
Log output exists: True

Output paths:
Raw: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\excel\inventory_raw.xlsx
Staging: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\excel\sample_excel_staging.csv
Clean: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\excel\sample_excel_clean.csv
Log: F:\data\new\quanskill\DataVision_Duy\week2\logs\excel_ingestion_log.json


## 19. Summary

```text
data/raw/excel/inventory_raw.xlsx
data/staging/excel/sample_excel_staging.csv
data/clean/excel/sample_excel_clean.csv
logs/excel_ingestion_log.json
```